# LM Evaluation Harness: Standard Benchmarks

## Why This Matters

Training loss ≠ model quality. A model can overfit to training data, lose general capabilities,
or improve on task metrics while degrading on reasoning.

The **lm-evaluation-harness** by EleutherAI powers the HuggingFace Open LLM Leaderboard
and is used in virtually every serious LLM paper. It evaluates models on 60+ standardized benchmarks:

| Benchmark | What it Tests | Format |
|-----------|--------------|--------|
| MMLU | Knowledge across 57 subjects | Multiple choice |
| HellaSwag | Common sense reasoning | Sentence completion |
| TruthfulQA | Factual accuracy | Multiple choice |
| GSM8K | Grade school math | Open generation |
| HumanEval | Python coding | Code generation |
| ARC-Easy/Challenge | Science questions | Multiple choice |
| WinoGrande | Commonsense pronoun resolution | Fill in blank |

This notebook shows how to evaluate your fine-tuned models on these benchmarks.

In [ ]:
# Install lm-evaluation-harness
!pip install lm-eval -q 2>&1 | tail -3
import lm_eval; print(f"lm-eval {lm_eval.__version__}")

In [ ]:
# Quick evaluation on a small task to verify setup
# Using arc_easy (ARC Easy — multiple choice science questions, fast to run)
!lm_eval \
    --model hf \
    --model_args pretrained=Qwen/Qwen3-0.6B,dtype=float16 \
    --tasks arc_easy \
    --num_fewshot 0 \
    --batch_size 8 \
    --limit 100 \
    --output_path /tmp/eval_results \
    2>&1 | grep -E "arc_easy|acc|error" | head -20

In [ ]:
# Load and display results
import json, os, glob

result_files = glob.glob("/tmp/eval_results/**/*.json", recursive=True)
if result_files:
    with open(result_files[0]) as f:
        results = json.load(f)
    
    print("=== Evaluation Results ===")
    for task, metrics in results.get("results", {}).items():
        print(f"\n{task}:")
        for metric, value in metrics.items():
            if isinstance(value, (int, float)):
                print(f"  {metric}: {value:.4f}")
else:
    print("No result files found — check if evaluation ran correctly")

In [ ]:
# Programmatic evaluation (Python API)
import lm_eval
import torch

print("Evaluating Qwen3-0.6B on arc_easy (100 examples)...")

results = lm_eval.simple_evaluate(
    model="hf",
    model_args="pretrained=Qwen/Qwen3-0.6B,dtype=float16",
    tasks=["arc_easy"],
    num_fewshot=0,
    batch_size=8,
    limit=100,  # use full dataset in production
)

task_results = results["results"]
for task, metrics in task_results.items():
    print(f"\n{task}:")
    for k, v in sorted(metrics.items()):
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")

## Evaluating Your Fine-Tuned Model

After fine-tuning, compare base vs fine-tuned:

```python
# 1. Evaluate base model
base_results = lm_eval.simple_evaluate(
    model="hf",
    model_args="pretrained=Qwen/Qwen3-0.6B,dtype=float16",
    tasks=["mmlu", "arc_easy", "hellaswag"],
    num_fewshot=0,
    batch_size=8,
)

# 2. Evaluate fine-tuned (load adapter)
from peft import PeftModel
from transformers import AutoModelForCausalLM
base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B", dtype=torch.float16)
ft = PeftModel.from_pretrained(base, "./my-adapter")
ft = ft.merge_and_unload()  # merge for clean eval
ft.save_pretrained("/tmp/merged-model")

ft_results = lm_eval.simple_evaluate(
    model="hf",
    model_args="pretrained=/tmp/merged-model,dtype=float16",
    tasks=["mmlu", "arc_easy", "hellaswag"],
    num_fewshot=0,
    batch_size=8,
)

# 3. Compare
for task in ["mmlu", "arc_easy", "hellaswag"]:
    base_acc = base_results["results"][task].get("acc,none", 0)
    ft_acc   = ft_results["results"][task].get("acc,none", 0)
    delta    = ft_acc - base_acc
    print(f"{task}: base={base_acc:.3f} → ft={ft_acc:.3f} (Δ{delta:+.3f})")
```

## Key Benchmarks to Always Run

1. **MMLU** (5-shot): General knowledge — did fine-tuning hurt broad capabilities?
2. **GSM8K** (8-shot): Math reasoning — did alignment improve/hurt reasoning?
3. **HellaSwag** (10-shot): Common sense — quick sanity check
4. **TruthfulQA** (0-shot): Did alignment improve factual accuracy?

Catastrophic forgetting shows as drops in MMLU/HellaSwag even when task-specific metrics improve.